In [33]:
import pandas as pd

df = pd.read_csv("clean_diabetic_data.csv")
df.shape

(101766, 47)

In [34]:
(df == "?").sum().sort_values(ascending=False).head(10)

diag_3                 1423
patient_nbr               0
encounter_id              0
gender                    0
age                       0
admission_type_id         0
race                      0
admission_source_id       0
time_in_hospital          0
num_lab_procedures        0
dtype: int64

In [35]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [36]:
df.isna().sum().sort_values(ascending=False).head(10)

max_glu_serum          96420
A1Cresult              84748
diag_2                  2894
diag_1                  1666
encounter_id               0
patient_nbr                0
race                       0
admission_source_id        0
time_in_hospital           0
num_lab_procedures         0
dtype: int64

In [37]:
df["max_glu_serum"].unique()

<StringArray>
[nan, '>200', 'Norm', '>300']
Length: 4, dtype: str

In [38]:
df["max_glu_serum"] = df["max_glu_serum"].fillna("None")
df["A1Cresult"] = df["A1Cresult"].fillna("None")

In [39]:
df[["max_glu_serum", "A1Cresult"]].isna().sum()

max_glu_serum    0
A1Cresult        0
dtype: int64

In [40]:
def to_binary_target(value):
    if value == "<30":
        readmitted = 1
    else:
        readmitted = 0 
    return readmitted
df["readmitted_30d"] = df["readmitted"].apply(to_binary_target)

In [41]:
df["readmitted_30d"].value_counts()

readmitted_30d
0    90409
1    11357
Name: count, dtype: int64

In [42]:
df["diag_1"].nunique()

697

In [43]:
df["diag_3"].astype(str).str.startswith(("V", "E")).sum()

np.int64(5058)

In [44]:
df["diag_1"] = df["diag_1"].astype(str)
df["diag_2"] = df["diag_2"].astype(str)

In [45]:
diag_1_fixed = pd.read_csv("clean_diabetic_data.csv", usecols=["diag_1"], dtype=str)
diag_1_fixed["diag_1"].astype(str).str.startswith(("V", "E")).sum()

np.int64(0)

In [46]:
diag_2_fixed = pd.read_csv("clean_diabetic_data.csv", usecols=["diag_2"], dtype=str)
diag_2_fixed["diag_2"].astype(str).str.startswith(("V", "E")).sum()

np.int64(0)

In [48]:
df.shape

(101766, 48)

In [49]:
def categorize_diagnosis(code):
    if pd.isna(code):
        return "Missing"
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    
    code_num = float(code)
    
    if 250 <= code_num < 251:
        return "Diabetes"
    elif 390 <= code_num <= 459 or code_num == 785:
        return "Circulatory"
    elif 460 <= code_num <= 519 or code_num == 786:
        return "Respiratory"
    elif 520 <= code_num <= 579 or code_num == 787:
        return "Digestive"
    elif 800 <= code_num <= 999:
        return "Injury"
    elif 710 <= code_num <= 739:
        return "Musculoskeletal"
    elif 580 <= code_num <= 629 or code_num == 788:
        return "Genitourinary"
    elif 140 <= code_num <= 239:
        return "Neoplasms"
    else:
        return "Other"

In [50]:
df["diag_1_group"] = df["diag_1"].apply(categorize_diagnosis)
df["diag_1_group"].value_counts()

diag_1_group
Circulatory        30437
Other              16527
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Missing             1666
Name: count, dtype: int64

In [58]:
df.shape

(101766, 47)

In [53]:
df["diag_3"] = df["diag_3"].replace("?", pd.NA)
df["diag_3_group"] = df["diag_3"].apply(categorize_diagnosis)

In [54]:
df["diag_2_group"] = df["diag_2"].apply(categorize_diagnosis)
df["diag_3_group"] = df["diag_3"].apply(categorize_diagnosis)

In [56]:
df["diag_3_group"].value_counts()

diag_3_group
Circulatory        30306
Other              29195
Diabetes           17157
Respiratory         7358
Genitourinary       6680
Digestive           3930
Injury              1946
Musculoskeletal     1915
Neoplasms           1856
Missing             1423
Name: count, dtype: int64

In [57]:
df = df.drop(columns=["diag_1", "diag_2", "diag_3", "readmitted"])

In [59]:
df.select_dtypes(include="object").columns.tolist()

C:\Users\metth\AppData\Local\Temp\ipykernel_24444\3671663173.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include="object").columns.tolist()


['race',
 'gender',
 'age',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'diag_1_group',
 'diag_2_group',
 'diag_3_group']

In [60]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.shape

C:\Users\metth\AppData\Local\Temp\ipykernel_24444\2985986009.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns.tolist()


(101766, 113)

In [70]:
from sklearn.model_selection import GroupShuffleSplit

X = df_encoded.drop(columns=["readmitted_30d", "encounter_id", "patient_nbr"])
y = df_encoded["readmitted_30d"]
groups = df_encoded["patient_nbr"]

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(X_train.shape, X_test.shape)

(79541, 110) (19802, 110)


In [71]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (im

In [72]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(10)

number_inpatient            0.544256
discharge_disposition_id    0.216378
number_emergency            0.034898
number_diagnoses            0.027958
diag_1_group_Missing        0.017166
time_in_hospital            0.015780
num_medications             0.013143
num_lab_procedures          0.012203
diag_2_group_Neoplasms      0.009894
diag_3_group_Neoplasms      0.009351
dtype: float64

In [73]:
from sklearn.metrics import roc_auc_score

y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC: {auc:.3f}")


AUC: 0.663


In [74]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC: {auc:.3f}")

AUC: 0.663


In [75]:
df["discharge_disposition_id"].value_counts().head(15)

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
Name: count, dtype: int64

In [76]:

death_hospice_codes = [11, 13, 14, 19, 20, 21]
df_encoded = df_encoded[~df["discharge_disposition_id"].isin(death_hospice_codes)]

C:\Users\metth\AppData\Local\Temp\ipykernel_24444\1439329498.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_encoded = df_encoded[~df["discharge_disposition_id"].isin(death_hospice_codes)]


In [77]:
print(df.shape, df_encoded.shape)

(101766, 47) (99343, 113)


In [78]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
auc_log = roc_auc_score(y_test, log_reg.predict_proba(X_test)[:, 1])

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print(f"Logistic Regression AUC: {auc_log:.3f}")
print(f"Random Forest AUC:       {auc_rf:.3f}")
print(f"Gradient Boosting AUC:   {auc:.3f}")

C:\Users\metth\readmission-predictor\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression AUC: 0.648
Random Forest AUC:       0.632
Gradient Boosting AUC:   0.663


In [79]:
import joblib

joblib.dump(model, "readmission_model.pkl")
joblib.dump(X_train.columns.tolist(), "model_features.pkl")

['model_features.pkl']

In [80]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'clean_diabetic_data.csv', 'exploration.ipynb', 'model_features.pkl', 'readmission_model.pkl', 'venv']
